In [ ]:
#IMPORTS
from pynq import Overlay, allocate, PL, MMIO
from pynq.lib.video import VideoMode
import ipywidgets as widgets
from IPython.display import display
import cv2
import numpy as np
from enum import Enum

PL.reset()
overlay = Overlay("/home/xilinx/jupyter_notebooks/PynqZ2-ORB-Accelerator/VivadoHW.bit")
overlay.download()

dict_keys(['axi_gpio_0', 'axi_intc_0', 'axi_vdma_0', 'v_tpg_0', 'axis_switch_0', 'processing_system7_0'])


In [ ]:
#SETUP AND DEFINES
width, height = 1280, 720
tpg = overlay.v_tpg_0
vdma = overlay.axi_vdma_0
gpios = overlay.axi_gpio_0
image_widget = widgets.Image(format='jpeg', width=1280, height=720)
stream_switch = overlay.axis_switch_0

class VideoSource(Enum):
    HDMI = 0
    TPG_COLOR_BAR_BOX = 1
    TPG_GRID_BOX = 2


selected_source = VideoSource.TPG_COLOR_BAR_BOX

In [20]:
#MAIN
def setupVideoSource(new_src: VideoSource):
    global selected_source 
    selected_source = new_src
    stream_switch.write(0x040, int(selected_source != VideoSource.HDMI))
    stream_switch.write(0x000, 0x2)

    tpg.write(0x0000, 0x0)
    tpg.write(0x0010, height)
    tpg.write(0x0018, width)
    tpg.write(0x0000, 0x0)

    if selected_source == VideoSource.TPG_COLOR_BAR_BOX:
        tpg.write(0x0020, 9); tpg.write(0x0028, 1); tpg.write(0x0038, 8)
        tpg.write(0x00D8, 0); tpg.write(0x0058, 150); tpg.write(0x0078, 200)
        tpg.write(0x0080, 200); tpg.write(0x0088, 50); tpg.write(0x0090, 150)
        tpg.write(0x0000, 0x81)
    elif selected_source == VideoSource.TPG_GRID_BOX:
        tpg.write(0x0020, 11); tpg.write(0x0028, 1); tpg.write(0x0038, 8)
        tpg.write(0x00D8, 0); tpg.write(0x0058, 150); tpg.write(0x0078, 200)
        tpg.write(0x0080, 200); tpg.write(0x0088, 50); tpg.write(0x0090, 150)
        tpg.write(0x0000, 0x81)

setupVideoSource(selected_source)
vdma.readchannel.mode = VideoMode(width, height, 24)
vdma.readchannel.start()

display(image_widget)

print("Streaming via VDMA interrupts...")
old_button_value = gpios.read(0)

try:
    while True:
        #READ AND DISPLAY FRAME
        frame = vdma.readchannel.readframe()
        frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        _, jpeg = cv2.imencode('.jpg', frame_bgr)
        image_widget.value = jpeg.tobytes()

        #DETECT BUTTON AND SWITCH VIDEO SOURCE
        button_value = gpios.read(0)
        if button_value != old_button_value and button_value > 0:
            next_val = (selected_source.value + 1) % len(VideoSource)
            setupVideoSource(VideoSource(next_val))

        old_button_value = button_value
        

except KeyboardInterrupt:
    print("\nStopping stream...")
finally:
    tpg.write(0x00, 0x0)
    vdma.readchannel.stop()
    print("Stream stopped cleanly.")

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

Streaming via VDMA interrupts...

Stopping stream...
Stream stopped cleanly.
